# Proteome exploration with ESMC embeddings

A **single** scanpy graph drives everything: a KNN graph on the ESMC embeddings, one UMAP layout, and Leiden clusters — same neighbors graph, fixed seed. SAE features are tested for enrichment per cluster (proteins as "cells", SAE features as "genes") and annotated with descriptions from the ESM Atlas.

**Prerequisites:** run the embedding step first, and `pip install -e "..[cluster]"` (scanpy, leidenalg, igraph). Needs `BASEROW_TOKEN` / `BIOHUB_API_TOKEN` in the env.

In [ ]:
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings

cfg = load_config("../config/octopus_chierchiae.yaml")
df = load_embeddings(cfg, prefer_cache=True)   # prefer_cache=False to pull from Baserow
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")
df.head()

## One graph: KNN → UMAP → Leiden

Defaults match the original UMAP (`n_neighbors=15`, `min_dist=0.1`, cosine). Tune **granularity**: `N_NEIGHBORS`/`MIN_DIST` shape the UMAP, `LEIDEN_RES` the cluster count.

In [ ]:
import scanpy as sc
from och_annotate.analysis import build_anndata, sae_enrichment, plot_umap

# ---- tunable parameters (defaults reproduce the original UMAP) ----
SEED        = 0
N_NEIGHBORS = 15        # KNN/UMAP granularity
MIN_DIST    = 0.1       # UMAP point spread
METRIC      = "cosine"
LEIDEN_RES  = 1.0       # clustering granularity (higher = more clusters)

adata = build_anndata(df)
sc.pp.neighbors(adata, use_rep="X_esmc", n_neighbors=N_NEIGHBORS, metric=METRIC, random_state=SEED)
sc.tl.umap(adata, min_dist=MIN_DIST, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RES, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)

meta_cols = [c for c in df.columns if c not in ("embedding", "sae_top_features")]
coords = df[meta_cols].copy().reset_index(drop=True)
coords["umap_0"] = adata.obsm["X_umap"][:, 0]
coords["umap_1"] = adata.obsm["X_umap"][:, 1]
coords["leiden"] = adata.obs["leiden"].to_numpy()
print(f"{adata.n_obs} proteins; {coords['leiden'].nunique()} Leiden clusters")

In [ ]:
# Interactive UMAP (scanpy graph) colored by chromosome
plot_umap(coords, color="chromosome", title=f"{cfg.name} — UMAP (chromosome)").show()

In [ ]:
# Same UMAP, colored by Leiden cluster
plot_umap(coords, color="leiden", title=f"{cfg.name} — UMAP (Leiden clusters)").show()

## SAE-feature enrichment per cluster

Wilcoxon rank-sum on the SAE activation matrix (marker-gene test, SAE features as genes). Each enriched feature is named via its **ESM Atlas** `label`.

In [ ]:
from och_annotate.atlas import fetch_all_features

# Per-cluster enriched SAE features (n per cluster); FDR in pvals_adj.
enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)

# Name every feature via the full ESM Atlas dictionary (one call, cached; no Biohub credits).
feat_dict = fetch_all_features(cache_path="../data/sae_feature_dictionary.parquet")
labels = dict(zip(feat_dict["feature"].astype(str), feat_dict["label"]))
enrich["label"] = enrich["sae_feature"].astype(str).map(labels)

n_clusters = enrich["leiden"].nunique()
print(f"{len(enrich)} enrichment rows across {n_clusters} clusters "
      f"({enrich['sae_feature'].nunique()} unique features); "
      f"dictionary has {len(feat_dict)} Atlas descriptions")

# Full per-cluster enrichment (with labels) written out for review.
enrich.to_csv("../data/cluster_sae_enrichment.csv", index=False)
print("Full table -> data/cluster_sae_enrichment.csv")

In [ ]:
import pandas as pd
from IPython.display import display

# Top-5 enriched SAE features per cluster, named via the Atlas.
top5 = (enrich.sort_values(["leiden", "scores"], ascending=[True, False])
              .groupby("leiden", observed=True).head(5)
              .assign(cluster=lambda d: d["leiden"].astype(int))
              .sort_values(["cluster", "scores"], ascending=[True, False]))
top5["rank"] = top5.groupby("cluster").cumcount() + 1
top5 = top5[["cluster", "rank", "sae_feature", "label", "scores", "pvals_adj"]]

# Group visually by cluster: the cluster label spans its 5 rows (multi-index),
# with alternating shading per cluster block.
def _shade(row):
    tint = "#eef3fa" if row.name[0] % 2 == 0 else "#ffffff"
    return [f"background-color: {tint}"] * len(row)

styled = (top5.set_index(["cluster", "rank"])
              .style
              .format({"scores": "{:.1f}", "pvals_adj": "{:.1e}"})
              .apply(_shade, axis=1)
              .set_properties(**{"text-align": "left"})
              .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
with pd.option_context("display.max_rows", None):
    display(styled)

# Dotplot of marker SAE features across clusters
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

### SAE feature descriptions (ESM Atlas)

`och_annotate.atlas.fetch_all_features()` pulls the **entire** SAE codebook dictionary (16,384 features: index, label, description) in a single call to the public ESM Atlas list endpoint (`biohub.ai/esm/protein/api/v1alpha1/features`), cached under `data/` and **not** charged against Biohub embedding credits. For richer per-feature metadata (summary, category, top-activating proteins) use `atlas.feature_info(idx)`. Browse features at https://biohub.ai/esm/protein/atlas .

### Other next steps
- Write `adata.obs["leiden"]` back to Baserow as a `leiden_cluster` column.
- Tune `LEIDEN_RES` (cluster granularity) and `N_NEIGHBORS` / `MIN_DIST` (UMAP).
- Enrichment sharpens as SAE coverage completes across the proteome.